# bn-weight-bias-init-pattern — ex2: verify BN at init is near-identity on already-normalized inputs

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `bn-weight-bias-init-pattern`. Running the final beacon cell reports progress against the `GAN: BN weight=1 bias=0 init` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: BN weight=1 bias=0 init` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bn-weight-bias-init-pattern`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bn-weight-bias-init-pattern"
DD_SUBTOPIC = "GAN: BN weight=1 bias=0 init"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## BN at init is a near-identity transform — deepening

Ex1 verified `weight ~ N(1, 0.02)` and `bias = 0` per-element. The deeper question: WHY those values? Answer: because the affine map then approximates the IDENTITY on already-normalized features.

BatchNorm in `.eval()` mode applies
```
y = (x - running_mean) / sqrt(running_var + eps) * weight + bias
```

If `running_mean ≈ 0`, `running_var ≈ 1` (the default), `weight ≈ 1`, `bias = 0`, then `y ≈ x`. The BN layer at init is essentially a pass-through, so a freshly-initialized DCGAN doesn't have wild scale/shift surprises in its forward pass.

**`std=0.02` is a tiny perturbation.** Big enough to break channel-wise symmetry. Small enough not to disturb the near-identity property.

### Exercise 2 — verify BN at init is near-identity on already-normalized inputs

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the BatchNorm-at-init contract by applying the DCGAN init to a fresh BN layer, feeding already-normalized inputs in eval mode, and verifying the output deviates from input by at most the std=0.02 init noise.
> Keywords: batchnorm, init, identity-at-init, scale-shift, near-identity
> ```

**KCs targeted:** `bn-weight-bias-init-pattern`, `bn-affine-near-identity`

Implement `ex2_apply_bn_init_and_check(channels, batch_size, n_spatial)` that builds a single `nn.BatchNorm2d(channels)` layer, applies the DCGAN init, runs an already-normalized input through it in eval mode, and returns a triple `(layer, x, y)`:

1. Build `layer = nn.BatchNorm2d(channels)`.
2. Apply the DCGAN BN init INLINE — directly on `layer` (don't call a previous-ex function):
   - `nn.init.normal_(layer.weight, 1.0, 0.02)`
   - `nn.init.zeros_(layer.bias)`
3. Build a fixed-shape standard-normal input `x = t.randn(batch_size, channels, n_spatial, n_spatial)`.
4. Switch the layer to eval mode (`layer.eval()`) so it uses `running_mean=0, running_var=1` (the defaults at init).
5. Run `y = layer(x)` under `t.no_grad()`.
6. Return `(layer, x, y)`.

The test cell then verifies that `y` is element-wise close to `x * gamma + 0` where `gamma` is the per-channel BN weight — i.e. the BN is exactly the affine map at init, no normalization is being applied because the running stats are the identity.

In [ ]:
def ex2_apply_bn_init_and_check(channels: int, batch_size: int, n_spatial: int):
    import torch.nn as nn
    layer = nn.BatchNorm2d(channels)
    nn.init.normal_(layer.weight, 1.0, 0.02)
    nn.init.zeros_(layer.bias)
    x = t.randn(batch_size, channels, n_spatial, n_spatial)
    layer.eval()
    with t.no_grad():
        y = layer(x)
    return layer, x, y


<details><summary>Solution</summary>

```python
def ex2_apply_bn_init_and_check(channels: int, batch_size: int, n_spatial: int):
    import torch.nn as nn
    layer = nn.BatchNorm2d(channels)
    nn.init.normal_(layer.weight, 1.0, 0.02)
    nn.init.zeros_(layer.bias)
    x = t.randn(batch_size, channels, n_spatial, n_spatial)
    layer.eval()
    with t.no_grad():
        y = layer(x)
    return layer, x, y
```

**Why the eval-mode forward is essential to the test.** In train mode, BN computes mean/var on the input batch itself (and the result of `y = (x - batch_mean) / batch_std * gamma + beta` is NOT `x * gamma` — it's `(normalized(x)) * gamma`, which destroys the original signal). Eval mode uses the running stats (default 0, 1 at init), making BN a pure affine map.

**Why DCGAN doesn't simply use `nn.init.ones_(layer.weight)`.** Setting every channel's gamma to exactly 1 would make the BN layer perfectly symmetric — every channel has the same scale. The `std=0.02` jitter breaks that symmetry so each channel can learn a slightly different scale during training.

**Numerical bound is `eps`-driven.** `BatchNorm2d.eps` defaults to `1e-5`, so the rescaling factor `1 / sqrt(running_var + eps)` = `1 / sqrt(1.00001) ≈ 0.999995`. The drift from pure-identity is at the eps-level — well under 1e-3 across all entries, matching invariant 3.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()